In [65]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [66]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-06-20,100.104828,101.113718,100.030093,100.637300
1,2016-06-21,100.422409,100.627926,100.048743,100.235580
2,2016-06-22,100.179527,101.066985,100.095456,100.478459
3,2016-06-23,101.608795,101.627482,100.534515,100.954885
4,2016-06-24,97.423744,99.488241,97.208884,97.993584
...,...,...,...,...,...
2509,2026-06-12,721.340027,724.010010,711.280029,717.609985
2510,2026-06-15,744.000000,744.760010,737.380005,738.099976
2511,2026-06-16,729.859985,744.219971,729.640015,742.250000
2512,2026-06-17,722.510010,735.679993,720.849976,735.190002


In [67]:
def VORTEX(data, n):
    pos_vm = abs(data['High'] - data['Low'].shift(1))
    neg_vm = abs(data['Low'] - data['High'].shift(1))
    pos_vm_sum = pos_vm.rolling(n).sum()
    neg_vm_sum = neg_vm.rolling(n).sum()
    meth1 = (data['High'] - data['Low'])
    meth2 = abs(data['High'] - data['Close'].shift(1))
    meth3 = abs(data['Low'] - data['Close'].shift(1))
    tr = abs(np.maximum(np.maximum(meth1, meth2), meth3))
    tr_sum = tr.rolling(n).sum()
    vibull = pos_vm_sum / tr_sum
    vibear = neg_vm_sum / tr_sum
    return vibull, vibear
    
df['VIBULL'], df['VIBEAR'] = VORTEX(df, 14)
df

Price,Date,Close,High,Low,Open,VIBULL,VIBEAR
0,2016-06-20,100.104828,101.113718,100.030093,100.637300,NaN,NaN
1,2016-06-21,100.422409,100.627926,100.048743,100.235580,NaN,NaN
2,2016-06-22,100.179527,101.066985,100.095456,100.478459,NaN,NaN
3,2016-06-23,101.608795,101.627482,100.534515,100.954885,NaN,NaN
4,2016-06-24,97.423744,99.488241,97.208884,97.993584,NaN,NaN
...,...,...,...,...,...,...,...
2509,2026-06-12,721.340027,724.010010,711.280029,717.609985,0.873014,0.895387
2510,2026-06-15,744.000000,744.760010,737.380005,738.099976,0.914652,0.906379
2511,2026-06-16,729.859985,744.219971,729.640015,742.250000,0.879448,0.920202
2512,2026-06-17,722.510010,735.679993,720.849976,735.190002,0.841836,0.970035


In [68]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data['VIBULL'].iloc[i-1] < data['VIBEAR'].iloc[i-1]) and (data['VIBULL'].iloc[i] > data['VIBEAR'].iloc[i]):
            signal[i] = 1
        elif (data['VIBULL'].iloc[i-1] > data['VIBEAR'].iloc[i-1]) and (data['VIBULL'].iloc[i] < data['VIBEAR'].iloc[i]):
            signal[i] = 2
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)
df

Price,Date,Close,High,Low,Open,VIBULL,VIBEAR,signal
0,2016-06-20,100.104828,101.113718,100.030093,100.637300,NaN,NaN,0
1,2016-06-21,100.422409,100.627926,100.048743,100.235580,NaN,NaN,0
2,2016-06-22,100.179527,101.066985,100.095456,100.478459,NaN,NaN,0
3,2016-06-23,101.608795,101.627482,100.534515,100.954885,NaN,NaN,0
4,2016-06-24,97.423744,99.488241,97.208884,97.993584,NaN,NaN,0
...,...,...,...,...,...,...,...,...
2509,2026-06-12,721.340027,724.010010,711.280029,717.609985,0.873014,0.895387,0
2510,2026-06-15,744.000000,744.760010,737.380005,738.099976,0.914652,0.906379,1
2511,2026-06-16,729.859985,744.219971,729.640015,742.250000,0.879448,0.920202,2
2512,2026-06-17,722.510010,735.679993,720.849976,735.190002,0.841836,0.970035,0


In [69]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2311
2     102
1     101
Name: count, dtype: int64


(2514, 10)

In [70]:
df.set_index('Date', inplace=True)

In [71]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.VIBULL, 
                         line=dict(color='lightseagreen', width=2),
                         name='Vortex Bull'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.VIBEAR, 
                         line=dict(color='red', width=2),
                         name='Vortex Bear'),
                         row=2, col=1)

fig.add_hline(y=1, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [72]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, sl=0.98*price, tp=1.12*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                #self.sell(size=0.99)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2513 [00:00<?, ?bar/s]

Start                     2016-06-20 00:00:00
End                       2026-06-18 00:00:00
Duration                   3650 days 00:00:00
Exposure Time [%]                    55.68815
Equity Final [$]                  380767.3425
Equity Peak [$]                  384760.28589
Commissions [$]                   21884.62545
Return [%]                          280.76734
Buy & Hold Return [%]               639.84443
Return (Ann.) [%]                    14.34168
Volatility (Ann.) [%]                12.71774
CAGR [%]                              9.67039
Sharpe Ratio                          1.12769
Sortino Ratio                         1.95047
Calmar Ratio                          0.73263
Alpha [%]                            122.0675
Beta                                  0.24803
Max. Drawdown [%]                   -19.57554
Avg. Drawdown [%]                    -2.12441
Max. Drawdown Duration      689 days 00:00:00
Avg. Drawdown Duration       32 days 00:00:00
# Trades                          

In [73]:
df['spread'] = (df['VIBULL'] - df['VIBEAR'])
df['spread_mom'] = (df['spread'] - df['spread'].shift(9))
df['smooth_mom'] = df['spread_mom'].rolling(8).mean()


In [74]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data['smooth_mom'].iloc[i] > data['smooth_mom'].iloc[i-1]) &\
        (data['smooth_mom'].iloc[i-1] < data['smooth_mom'].iloc[i-2]) &\
        (data['smooth_mom'].iloc[i] < 0):
            signal[i] = 1
        elif (data['smooth_mom'].iloc[i-1] > 0) and (data['smooth_mom'].iloc[i] < 0):
            signal[i] = 2
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)
df

Price,Close,High,Low,Open,VIBULL,VIBEAR,signal,long_entries,short_entries,spread,spread_mom,smooth_mom
Date,,,,,,,,,,,,
2016-06-20,100.104828,101.113718,100.030093,100.637300,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2016-06-21,100.422409,100.627926,100.048743,100.235580,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2016-06-22,100.179527,101.066985,100.095456,100.478459,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2016-06-23,101.608795,101.627482,100.534515,100.954885,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2016-06-24,97.423744,99.488241,97.208884,97.993584,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-12,721.340027,724.010010,711.280029,717.609985,0.873014,0.895387,0,NaN,NaN,-0.022372,-0.377330,-0.351968
2026-06-15,744.000000,744.760010,737.380005,738.099976,0.914652,0.906379,0,735.905245,NaN,0.008274,-0.515106,-0.409781
2026-06-16,729.859985,744.219971,729.640015,742.250000,0.879448,0.920202,0,NaN,745.708411,-0.040754,-0.502969,-0.449406


In [75]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2298
1     130
2      86
Name: count, dtype: int64


(2514, 12)

In [76]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.smooth_mom, 
                         line=dict(color='lightseagreen', width=2),
                         name='Smooth Mom'),
                         row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [77]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                #self.sell(size=0.99)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2513 [00:00<?, ?bar/s]

Start                     2016-06-20 00:00:00
End                       2026-06-18 00:00:00
Duration                   3650 days 00:00:00
Exposure Time [%]                    80.19093
Equity Final [$]                 510744.67115
Equity Peak [$]                  514522.93616
Commissions [$]                    17583.1728
Return [%]                          410.74467
Buy & Hold Return [%]               639.84443
Return (Ann.) [%]                    17.75773
Volatility (Ann.) [%]                23.40381
CAGR [%]                             11.91677
Sharpe Ratio                          0.75875
Sortino Ratio                         1.29739
Calmar Ratio                          0.61402
Alpha [%]                           -89.97383
Beta                                  0.78256
Max. Drawdown [%]                   -28.92035
Avg. Drawdown [%]                     -2.6937
Max. Drawdown Duration      476 days 00:00:00
Avg. Drawdown Duration       27 days 00:00:00
# Trades                          

In [78]:
def BANDS(data, n1, n2, n3):
    sv = data['smooth_mom']
    std = sv.rolling(n1).std()
    mid = sv.rolling(n1).mean()
    up = mid + std * n2
    lo = mid - std * n3
    return up, lo

df['upper'], df['lower'] = BANDS(df, 20, 2, 2)


In [79]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data['smooth_mom'].iloc[i-1] < data['lower'].iloc[i-1]) &\
        (data['smooth_mom'].iloc[i] > data['lower'].iloc[i]):
            signal[i] = 1
        elif (data['smooth_mom'].iloc[i-1] > data['upper'].iloc[i-1]) &\
        (data['smooth_mom'].iloc[i] < data['upper'].iloc[i]):
            signal[i] = 2
        elif (data['smooth_mom'].iloc[i] > data['smooth_mom'].iloc[i-1]) &\
        (data['smooth_mom'].iloc[i-1] < data['smooth_mom'].iloc[i-2]) &\
        (data['smooth_mom'].iloc[i] < 0):
            signal[i] = 3
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)
df

Price,Close,High,Low,Open,VIBULL,VIBEAR,signal,long_entries,short_entries,spread,spread_mom,smooth_mom,upper,lower
Date,,,,,,,,,,,,,,
2016-06-20,100.104828,101.113718,100.030093,100.637300,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-06-21,100.422409,100.627926,100.048743,100.235580,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-06-22,100.179527,101.066985,100.095456,100.478459,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-06-23,101.608795,101.627482,100.534515,100.954885,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-06-24,97.423744,99.488241,97.208884,97.993584,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-12,721.340027,724.010010,711.280029,717.609985,0.873014,0.895387,0,NaN,NaN,-0.022372,-0.377330,-0.351968,-0.049807,-0.368416
2026-06-15,744.000000,744.760010,737.380005,738.099976,0.914652,0.906379,0,NaN,NaN,0.008274,-0.515106,-0.409781,-0.060416,-0.393093
2026-06-16,729.859985,744.219971,729.640015,742.250000,0.879448,0.920202,0,NaN,NaN,-0.040754,-0.502969,-0.449406,-0.069385,-0.422413


In [83]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def more_long_entries(x):
    offset = 0.002
    if x['signal']==3:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['more_long_entries'] = df.apply(lambda x: more_long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2315
3     120
1      43
2      36
Name: count, dtype: int64


(2514, 15)

In [84]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['more_long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="blue"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.smooth_mom, 
                         line=dict(color='white', width=2),
                         name='Smooth Mom'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.upper, 
                         line=dict(color='red', width=1),
                         name='Smooth Mom'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.lower, 
                         line=dict(color='lightseagreen', width=1),
                         name='Smooth Mom'),
                         row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [88]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99)

        elif self.signal==3: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.02*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99, sl=1.02*price, tp=0.9*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2513 [00:00<?, ?bar/s]

Start                     2016-06-20 00:00:00
End                       2026-06-18 00:00:00
Duration                   3650 days 00:00:00
Exposure Time [%]                    80.86714
Equity Final [$]                 845526.92347
Equity Peak [$]                  851787.09864
Commissions [$]                    28670.1723
Return [%]                          745.52692
Buy & Hold Return [%]               639.84443
Return (Ann.) [%]                    23.86084
Volatility (Ann.) [%]                25.12651
CAGR [%]                             15.88038
Sharpe Ratio                          0.94963
Sortino Ratio                         1.75403
Calmar Ratio                           1.0156
Alpha [%]                           329.52337
Beta                                  0.65016
Max. Drawdown [%]                   -23.49434
Avg. Drawdown [%]                    -2.77988
Max. Drawdown Duration      456 days 00:00:00
Avg. Drawdown Duration       25 days 00:00:00
# Trades                          